# Pipeline del agente paso a paso

Consulta → interpretación → selección de función → ejecución sobre el grafo → respuesta.

Usa el mismo paquete `agente/` que la API. Con `NEO4J_URI` en el entorno trabaja contra Neo4j;
si no, contra el catálogo en memoria (mismo contrato, mismas pruebas).

In [1]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))

from agente.datos import leer_productos, leer_vendedores
from agente.catalogo import CatalogoMemoria

if os.getenv("NEO4J_URI"):
    from agente.catalogo_neo4j import CatalogoNeo4j
    catalogo = CatalogoNeo4j(os.environ["NEO4J_URI"], os.getenv("NEO4J_USER", "neo4j"), os.getenv("NEO4J_PASSWORD", ""))
    catalogo.cargar(leer_productos(), leer_vendedores())
    print("Catálogo: Neo4j")
else:
    catalogo = CatalogoMemoria(leer_productos(), leer_vendedores())
    print("Catálogo: memoria")
print(len(catalogo.productos()), "productos,", len(catalogo.vendedores()), "vendedores")

Catálogo: memoria
27 productos, 3 vendedores


## 1. Tabla de funciones

Cada función tiene una descripción y ejemplos de cómo se pide.

In [2]:
from agente.funciones import FUNCIONES
for f in FUNCIONES:
    print(f"{f['nombre']:<24} {len(f['ejemplos'])} ejemplos  {f['descripcion']}")

buscar_producto          9 ejemplos  Busca productos por nombre o características y devuelve precio, vendedor y ubicación.
buscar_vendedor          8 ejemplos  Encuentra vendedores por ciudad, especialidad, calificación o producto que venden.
verificar_stock          8 ejemplos  Verifica disponibilidad y unidades en stock de un producto.
comparar_precios         7 ejemplos  Compara precios entre productos parecidos e indica el más barato y el más caro.
crear_pedido             8 ejemplos  Agrega un producto al carrito para hacer un pedido.
consultar_categoria      7 ejemplos  Lista las categorías o los productos de una categoría con su rango de precios.
buscar_por_ubicacion     6 ejemplos  Productos almacenados y vendedores ubicados en una ciudad.
obtener_recomendaciones  7 ejemplos  Sugiere productos complementarios o que otros compraron junto con uno.
saludo                   7 ejemplos  Saludos y conversación sin pedido concreto.


## 2. Interpretación

Se extraen términos de producto, categoría, ciudad y cantidad. Para elegir la función se usa una
*plantilla* donde los productos, ciudades y números quedan como marcadores: así dos preguntas
con la misma forma se parecen aunque nombren productos distintos.

In [3]:
from agente.agente import Agente
from agente.redaccion import Redactor

agente = Agente(catalogo, Redactor(clave=""))
for q in ["¿Cuánto cuesta el iPhone 14?", "¿Cuántos iPhone 14 hay disponibles?", "Quiero 2 cables HDMI", "¿Quién vende smartphones en Lima?"]:
    e = agente.extractor.extraer(q)
    print(f"{q}\n   plantilla: {e.plantilla}\n   términos: {e.terminos}  ciudad: {e.ciudad}  cantidad: {e.cantidad}")

¿Cuánto cuesta el iPhone 14?
   plantilla: cuanto cuesta el PRODUCTO
   términos: ['iphone', '14']  ciudad: None  cantidad: None
¿Cuántos iPhone 14 hay disponibles?
   plantilla: cuantos PRODUCTO hay disponibles
   términos: ['iphone', '14']  ciudad: None  cantidad: None
Quiero 2 cables HDMI
   plantilla: quiero NUM PRODUCTO
   términos: ['cable', 'hdmi']  ciudad: None  cantidad: 2
¿Quién vende smartphones en Lima?
   plantilla: quien vende PRODUCTO en CIUDAD
   términos: ['smartphone']  ciudad: Lima  cantidad: None


## 3. Selección de función

Similitud coseno entre la plantilla y cada ejemplo (TF-IDF de n-gramas de caracteres); el puntaje
de una función es su mejor ejemplo. Por debajo de 0,3 el agente pide reformular.

In [4]:
for q in ["¿Cuánto cuesta el iPhone 14?", "¿Cuántos iPhone 14 hay disponibles?", "¿Qué smartphone es más barato?", "zxcv qwerty"]:
    el = agente.enrutador.elegir(agente.extractor.extraer(q).plantilla)
    top = ", ".join(f"{f} {s:.2f}" for f, s in list(el.puntajes.items())[:3])
    print(f"{q}\n   -> {el.funcion if el.segura else 'aclarar'}  ({top})")

¿Cuánto cuesta el iPhone 14?
   -> buscar_producto  (buscar_producto 1.00, comparar_precios 0.64, verificar_stock 0.36)
¿Cuántos iPhone 14 hay disponibles?
   -> verificar_stock  (verificar_stock 1.00, buscar_por_ubicacion 0.70, buscar_producto 0.36)
¿Qué smartphone es más barato?
   -> comparar_precios  (comparar_precios 1.00, obtener_recomendaciones 0.32, buscar_producto 0.28)
zxcv qwerty
   -> aclarar  (buscar_producto 0.22, buscar_por_ubicacion 0.18, consultar_categoria 0.16)


Precisión sobre 33 preguntas que no están en los ejemplos:

In [5]:
sys.path.insert(0, os.path.abspath("../tests"))
from consultas_etiquetadas import CONSULTAS

aciertos = sum(agente.enrutador.elegir(agente.extractor.extraer(q).plantilla).funcion == f for q, f in CONSULTAS)
print(f"{aciertos}/{len(CONSULTAS)} = {aciertos / len(CONSULTAS):.0%}")

33/33 = 100%


## 4. Grafo del agente (LangGraph) y respuesta

In [6]:
for q in ["¿Qué smartphone es más barato?", "¿Qué me recomiendas con una laptop?", "Quiero 2 iPhone 14 Pro", "Hola"]:
    r = agente.responder(q)
    print(q)
    for paso in r["traza"]:
        print(f"   {paso['paso']:<11} {paso['detalle']}")
    print("   =>", r["respuesta"], "\n")

¿Qué smartphone es más barato?
   interpretar plantilla «que PRODUCTO es mas barato»; términos ['smartphone']
   enrutar     tf-idf de caracteres: comparar_precios 1.00, obtener_recomendaciones 0.32, buscar_producto 0.28
   ejecutar    comparar_precios: 9 fila(s)
   redactar    plantilla
   => El más barato es Xiaomi Redmi Note 13 ($299.99) y el más caro iPhone 15 ($1,099.99): $800.00 de diferencia. 

¿Qué me recomiendas con una laptop?
   interpretar plantilla «que me recomiendas con NUM PRODUCTO»; términos ['laptop']; cantidad 1
   enrutar     tf-idf de caracteres: obtener_recomendaciones 1.00, crear_pedido 0.36, buscar_vendedor 0.32
   ejecutar    obtener_recomendaciones: 3 fila(s)
   redactar    plantilla
   => Con Laptop Acer Swift 5 le recomiendo: Mouse Logitech MX a $79.99; Teclado Mecanico a $129.99; Cable HDMI 2.1 a $19.99. 

Quiero 2 iPhone 14 Pro
   interpretar plantilla «quiero NUM PRODUCTO»; términos ['iphone', '14', 'pro']; cantidad 2
   enrutar     tf-idf de caracteres: 

## 5. Pedido

El chat solo propone la línea del carrito; el pedido se confirma aparte y el precio lo pone el catálogo.
Si no alcanza el stock, no se descuenta nada.

In [7]:
from agente.modelo import SinStock

pedido = catalogo.crear_pedido("Ana", [(2, 2), (14, 1)])
print(pedido.id, [(l.producto, l.cantidad, l.precio_unitario) for l in pedido.lineas], "total", pedido.total)
try:
    catalogo.crear_pedido("Luis", [(20, 9)])
except SinStock as e:
    print("Rechazado:", e)

PED-00001 [('iPhone 14 Pro', 2, 999.99), ('Cable HDMI 2.1', 1, 19.99)] total 2019.97
Rechazado: Solo quedan 8 unidades de Laptop MSI Creator 15
